# Sub‑step 1: Load data & schema check
## Goal: 
Confirm the data loaded correctly and all required tables and columns exist.
## What this code checks
- Excel file can be read
- All expected tables are present
- Table shapes (rows, columns) are visible
- No structural issues block further analysis

In [1]:
import pandas as pd

file_path = "maplecare_dw.xlsx"

print("STEP 4 – SUB‑STEP 1: LOAD DATA & SCHEMA CHECK\n")

# 1. Check if Excel file can be read
try:
    sheets = pd.read_excel(file_path, sheet_name=None, engine="openpyxl")
    print("✅ Excel file read successfully.")
except Exception as e:
    raise RuntimeError(f"❌ Failed to read Excel file: {e}")

# 2. Check expected tables
expected_tables = [
    "fact_claim_line",
    "dim_date",
    "dim_patient",
    "dim_provider",
    "dim_payer",
    "dim_facility",
    "dim_denial_reason",
    "dim_cpt",
    "dim_icd"
]

missing_tables = [t for t in expected_tables if t not in sheets]

if missing_tables:
    raise ValueError(f"❌ Missing tables detected: {missing_tables}")
else:
    print("✅ All expected tables are present.")

# 3. Print table shapes (rows, columns)
print("\n📊 Table Shapes:")
for table in expected_tables:
    df = sheets[table]
    print(f"   - {table}: {df.shape[0]} rows, {df.shape[1]} columns")

# 4. Confirm structural usability
print("\n✅ Schema validation completed.")
print("✅ No structural issues blocking further analysis.")

STEP 4 – SUB‑STEP 1: LOAD DATA & SCHEMA CHECK

✅ Excel file read successfully.
✅ All expected tables are present.

📊 Table Shapes:
   - fact_claim_line: 120000 rows, 19 columns
   - dim_date: 1096 rows, 7 columns
   - dim_patient: 6000 rows, 2 columns
   - dim_provider: 80 rows, 2 columns
   - dim_payer: 5 rows, 3 columns
   - dim_facility: 4 rows, 2 columns
   - dim_denial_reason: 5 rows, 3 columns
   - dim_cpt: 8 rows, 4 columns
   - dim_icd: 5 rows, 4 columns

✅ Schema validation completed.
✅ No structural issues blocking further analysis.


# Sub‑step 2: Data quality profiling (PKs, nulls, duplicates)
## Goal:
Confirm that counts and financial values won’t be inflated or broken.
## ✅ What this code checks

- Primary key uniqueness
- Missing values in critical columns
- Duplicate claim lines

In [2]:
print("\nSTEP 4 – SUB‑STEP 2: DATA QUALITY PROFILING (UPDATED FOR ENRICHED DATASET)\n")

# -----------------------------
# Assign tables
# -----------------------------
fact = sheets["fact_claim_line"]
dim_date = sheets["dim_date"]
dim_patient = sheets["dim_patient"]
dim_provider = sheets["dim_provider"]
dim_payer = sheets["dim_payer"]
dim_facility = sheets["dim_facility"]
dim_denial = sheets["dim_denial_reason"]
dim_cpt = sheets["dim_cpt"]
dim_icd = sheets["dim_icd"]

# -----------------------------
# 1. Primary Key Uniqueness Checks
# -----------------------------
print("🔑 Primary Key Uniqueness Checks:")

pk_checks = {
    "fact_claim_line.claim_line_id_nat": fact["claim_line_id_nat"].is_unique,
    "dim_date.date_sk": dim_date["date_sk"].is_unique,
    "dim_patient.patient_sk": dim_patient["patient_sk"].is_unique,
    "dim_provider.provider_sk": dim_provider["provider_sk"].is_unique,
    "dim_payer.payer_sk": dim_payer["payer_sk"].is_unique,
    "dim_facility.facility_sk": dim_facility["facility_sk"].is_unique,
    "dim_cpt.cpt_sk": dim_cpt["cpt_sk"].is_unique,
    "dim_icd.icd_sk": dim_icd["icd_sk"].is_unique,
    "dim_denial_reason.denial_reason_sk": dim_denial["denial_reason_sk"].is_unique,
}

for k, v in pk_checks.items():
    status = "✅ PASS" if v else "❌ FAIL"
    print(f"   - {k}: {status}")

# -----------------------------
# 2. Null Checks on Critical Columns
# -----------------------------
print("\n🧩 Null Checks on Critical Columns:")

critical_columns = {
    "fact_claim_line": [
        "claim_line_id_nat",
        "service_date_sk",
        "patient_sk",
        "provider_sk",
        "facility_sk",
        "payer_sk",
        "cpt_sk",
        "icd_sk",
        "submit_lag_days",
        "has_prior_authorization",
        "coding_completeness_score",
        "charge_amount",
        "allowed_amount",
        "paid_amount",
    ],
    "dim_patient": ["patient_sk"],
    "dim_provider": ["provider_sk", "provider_name"],
    "dim_payer": ["payer_sk", "payer_name"],
    "dim_facility": ["facility_sk", "facility_name"],
    "dim_cpt": ["cpt_sk", "cpt_code"],
    "dim_icd": ["icd_sk", "icd_code"],
    "dim_denial_reason": ["denial_reason_sk", "denial_reason"],
}

table_map = {
    "fact_claim_line": fact,
    "dim_patient": dim_patient,
    "dim_provider": dim_provider,
    "dim_payer": dim_payer,
    "dim_facility": dim_facility,
    "dim_cpt": dim_cpt,
    "dim_icd": dim_icd,
    "dim_denial_reason": dim_denial,
}

for table, cols in critical_columns.items():
    df = table_map[table]
    for col in cols:
        null_pct = df[col].isna().mean() * 100
        status = "✅ OK" if null_pct < 0.1 else "⚠️ REVIEW"
        print(f"   - {table}.{col}: {null_pct:.2f}% nulls {status}")

# -----------------------------
# 3. Duplicate Claim‑Line Check
# -----------------------------
duplicate_count = fact.duplicated(subset=["claim_line_id_nat"]).sum()

print("\n📌 Duplicate Check:")
if duplicate_count == 0:
    print("   ✅ No duplicate claim_line_id_nat found.")
else:
    print(f"   ❌ {duplicate_count} duplicate claim lines detected.")

# -----------------------------
# 4. Referential Integrity (NEW, VERY IMPORTANT)
# -----------------------------
print("\n🔗 Referential Integrity Checks:")

fk_checks = {
    "cpt_sk missing in dim_cpt": ~fact["cpt_sk"].isin(dim_cpt["cpt_sk"]),
    "icd_sk missing in dim_icd": ~fact["icd_sk"].isin(dim_icd["icd_sk"]),
    "payer_sk missing in dim_payer": ~fact["payer_sk"].isin(dim_payer["payer_sk"]),
    "facility_sk missing in dim_facility": ~fact["facility_sk"].isin(dim_facility["facility_sk"]),
}

for desc, mask in fk_checks.items():
    count = mask.sum()
    status = "✅ PASS" if count == 0 else f"❌ FAIL ({count} rows)"
    print(f"   - {desc}: {status}")

print("\n✅ Data quality profiling completed for enriched dataset.")


STEP 4 – SUB‑STEP 2: DATA QUALITY PROFILING (UPDATED FOR ENRICHED DATASET)

🔑 Primary Key Uniqueness Checks:
   - fact_claim_line.claim_line_id_nat: ✅ PASS
   - dim_date.date_sk: ✅ PASS
   - dim_patient.patient_sk: ✅ PASS
   - dim_provider.provider_sk: ✅ PASS
   - dim_payer.payer_sk: ✅ PASS
   - dim_facility.facility_sk: ✅ PASS
   - dim_cpt.cpt_sk: ✅ PASS
   - dim_icd.icd_sk: ✅ PASS
   - dim_denial_reason.denial_reason_sk: ✅ PASS

🧩 Null Checks on Critical Columns:
   - fact_claim_line.claim_line_id_nat: 0.00% nulls ✅ OK
   - fact_claim_line.service_date_sk: 0.00% nulls ✅ OK
   - fact_claim_line.patient_sk: 0.00% nulls ✅ OK
   - fact_claim_line.provider_sk: 0.00% nulls ✅ OK
   - fact_claim_line.facility_sk: 0.00% nulls ✅ OK
   - fact_claim_line.payer_sk: 0.00% nulls ✅ OK
   - fact_claim_line.cpt_sk: 0.00% nulls ✅ OK
   - fact_claim_line.icd_sk: 0.00% nulls ✅ OK
   - fact_claim_line.submit_lag_days: 0.00% nulls ✅ OK
   - fact_claim_line.has_prior_authorization: 0.00% nulls ✅ OK
   - fa

# Sub‑step 3: Date & Temporal Integrity Checks
This step validates that time‑based behavior in the claims data is realistic and usable for both:

- Revenue‑cycle analytics
- Machine‑learning models (denials are often time‑sensitive)

In healthcare claims, timing errors are a major denial driver, so this step is critical.

In [3]:
print("\nSTEP 4 – SUB‑STEP 4.3: DATE & TEMPORAL INTEGRITY CHECKS\n")

# ----------------------------------
# Assign tables
# ----------------------------------
fact = sheets["fact_claim_line"]
dim_date = sheets["dim_date"]

# ----------------------------------
# 1. Service Date Validity Check
# ----------------------------------
print("📅 Service Date Validity Checks:")

# Join to date dimension
service_dates = fact.merge(
    dim_date[["date_sk", "full_date"]],
    left_on="service_date_sk",
    right_on="date_sk",
    how="left"
)

missing_service_dates = service_dates["full_date"].isna().sum()

if missing_service_dates == 0:
    print("   ✅ All service_date_sk values match dim_date.")
else:
    print(f"   ❌ {missing_service_dates} service_date_sk values not found in dim_date.")

# Future date check
today = pd.Timestamp.today().normalize()
future_service_dates = (service_dates["full_date"] > today).sum()

if future_service_dates == 0:
    print("   ✅ No future service dates detected.")
else:
    print(f"   ❌ {future_service_dates} future service dates detected.")

# ----------------------------------
# 2. Submission Lag Integrity
# ----------------------------------
print("\n⏱ Submission Lag (submit_lag_days) Analysis:")

negative_lag_count = (fact["submit_lag_days"] < 0).sum()

if negative_lag_count == 0:
    print("   ✅ No negative submission lags.")
else:
    print(f"   ❌ {negative_lag_count} negative submission lags detected.")

# Summary statistics
lag_stats = fact["submit_lag_days"].describe()

print("\n   Submission Lag Summary (days):")
print(f"   - Min: {lag_stats['min']:.0f}")
print(f"   - Median: {lag_stats['50%']:.0f}")
print(f"   - Mean: {lag_stats['mean']:.1f}")
print(f"   - 95th percentile: {fact['submit_lag_days'].quantile(0.95):.0f}")
print(f"   - Max: {lag_stats['max']:.0f}")

# ----------------------------------
# 3. Late Submission Business Check
# ----------------------------------
print("\n🚨 Late Submission Risk Check:")

late_claims = (fact["submit_lag_days"] > 30).mean() * 100

print(f"   - Claims submitted after 30 days: {late_claims:.2f}%")

if late_claims < 25:
    print("   ✅ Late submissions within expected operational range.")
else:
    print("   ⚠️ High proportion of late submissions — potential denial risk driver.")

print("\n✅ Step 4.3 Date & Temporal Integrity Checks completed.")


STEP 4 – SUB‑STEP 4.3: DATE & TEMPORAL INTEGRITY CHECKS

📅 Service Date Validity Checks:
   ✅ All service_date_sk values match dim_date.
   ✅ No future service dates detected.

⏱ Submission Lag (submit_lag_days) Analysis:
   ✅ No negative submission lags.

   Submission Lag Summary (days):
   - Min: 0
   - Median: 8
   - Mean: 9.5
   - 95th percentile: 23
   - Max: 65

🚨 Late Submission Risk Check:
   - Claims submitted after 30 days: 1.39%
   ✅ Late submissions within expected operational range.

✅ Step 4.3 Date & Temporal Integrity Checks completed.


# Step 4 – Sub‑step 4.4: Distribution & Statistical Sanity Checks
This is the final EDA checkpoint before Feature Engineering (Step 5).
Its job is to answer one critical question:

Do numeric, categorical, and behavioral distributions look realistic and analytically usable?

If this step passes, you can confidently move forward.

In [4]:
print("\nSTEP 4 – SUB‑STEP 4.4: DISTRIBUTION & STATISTICAL SANITY CHECKS\n")

import numpy as np

# -----------------------------
# Assign tables
# -----------------------------
fact = sheets["fact_claim_line"]
dim_payer = sheets["dim_payer"]
dim_cpt = sheets["dim_cpt"]
dim_icd = sheets["dim_icd"]

# -----------------------------
# 1. Financial Distribution Checks
# -----------------------------
print("💰 Financial Amount Distribution Summary:")

amount_cols = [
    "charge_amount",
    "allowed_amount",
    "paid_amount",
    "denial_amount",
    "value_at_risk"
]

for col in amount_cols:
    desc = fact[col].describe()
    print(f"\n   {col}:")
    print(f"   - Min: {desc['min']:.2f}")
    print(f"   - Median: {desc['50%']:.2f}")
    print(f"   - Mean: {desc['mean']:.2f}")
    print(f"   - 95th percentile: {fact[col].quantile(0.95):.2f}")
    print(f"   - Max: {desc['max']:.2f}")

# Logical ordering check
invalid_pricing = (
    (fact["charge_amount"] < fact["allowed_amount"]) |
    (fact["allowed_amount"] < fact["paid_amount"])
).sum()

if invalid_pricing == 0:
    print("\n✅ Pricing hierarchy valid: charge ≥ allowed ≥ paid.")
else:
    print(f"\n❌ {invalid_pricing} rows violate pricing hierarchy.")

# -----------------------------
# 2. Denial Rate Sanity
# -----------------------------
print("\n📉 Denial Rate Sanity Checks:")

overall_denial_rate = fact["is_denied"].mean() * 100
print(f"   - Overall denial rate: {overall_denial_rate:.2f}%")

payer_denials = fact.merge(
    dim_payer[["payer_sk", "payer_name"]],
    on="payer_sk",
    how="left"
).groupby("payer_name")["is_denied"].mean() * 100

print("\n   Denial rate by payer:")
print(payer_denials.round(2))

# -----------------------------
# 3. Clinical & Operational Signals
# -----------------------------
print("\n🧬 Clinical & Operational Feature Checks:")

auth_rate = fact["has_prior_authorization"].mean() * 100
coding_score_desc = fact["coding_completeness_score"].describe()

print(f"   - Prior authorization present: {auth_rate:.2f}%")
print("   - Coding completeness score:")
print(f"     • Min: {coding_score_desc['min']:.2f}")
print(f"     • Median: {coding_score_desc['50%']:.2f}")
print(f"     • Mean: {coding_score_desc['mean']:.2f}")
print(f"     • Max: {coding_score_desc['max']:.2f}")

# -----------------------------
# 4. Outlier Presence (IQR)
# -----------------------------
print("\n📊 Outlier Presence Check (IQR method):")

for col in ["charge_amount", "submit_lag_days"]:
    Q1 = fact[col].quantile(0.25)
    Q3 = fact[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_pct = ((fact[col] < lower) | (fact[col] > upper)).mean() * 100
    print(f"   - {col}: {outlier_pct:.2f}% outliers (expected, not removed)")

print("\n✅ Step 4.4 Distribution & Statistical Sanity Checks completed.")


STEP 4 – SUB‑STEP 4.4: DISTRIBUTION & STATISTICAL SANITY CHECKS

💰 Financial Amount Distribution Summary:

   charge_amount:
   - Min: 133.00
   - Median: 1045.10
   - Mean: 4238.61
   - 95th percentile: 15256.28
   - Max: 16099.78

   allowed_amount:
   - Min: 77.46
   - Median: 612.75
   - Mean: 2882.93
   - 95th percentile: 10738.24
   - Max: 12517.61

   paid_amount:
   - Min: 59.79
   - Median: 479.44
   - Mean: 2492.02
   - 95th percentile: 9424.78
   - Max: 12033.00

   denial_amount:
   - Min: 0.00
   - Median: 0.00
   - Mean: 214.88
   - 95th percentile: 1643.10
   - Max: 3058.65

   value_at_risk:
   - Min: 32.28
   - Median: 265.81
   - Mean: 1746.59
   - 95th percentile: 6840.82
   - Max: 8938.53

✅ Pricing hierarchy valid: charge ≥ allowed ≥ paid.

📉 Denial Rate Sanity Checks:
   - Overall denial rate: 43.35%

   Denial rate by payer:
payer_name
MSP          41.21
OHIP         40.12
Private A    45.31
Private B    47.25
RAMQ         42.80
Name: is_denied, dtype: float64



# Sub‑step 5: Amount sanity checks & aggregates

## Goal:
Confirm that financial data follows healthcare billing logic and can be trusted for revenue analysis.
## This step checks:
- No negative monetary values
- Core billing rules are respected (paid ≤ allowed ≤ charge)
- High‑level financial totals are computable for dashboards


## ✅ What this code checks:

- Negative values in amount columns
- Logical violations in billing flow
- Aggregate financial metrics

In [5]:
print("\nSTEP 4 – SUB‑STEP 4.5: AMOUNT SANITY CHECKS & AGGREGATES\n")

# -----------------------------
# Assign fact table
# -----------------------------
fact = sheets["fact_claim_line"]

# -----------------------------
# 1. Negative Value Checks
# -----------------------------
print("💰 Negative Value Checks:")

amount_columns = [
    "charge_amount",
    "allowed_amount",
    "paid_amount",
    "denial_amount",
    "value_at_risk"
]

for col in amount_columns:
    neg_count = (fact[col] < 0).sum()
    status = "✅ OK" if neg_count == 0 else f"❌ {neg_count} negatives"
    print(f"   - {col}: {status}")

# -----------------------------
# 2. Pricing Hierarchy Validation
# -----------------------------
print("\n📐 Pricing Hierarchy Check:")

pricing_violations = (
    (fact["charge_amount"] < fact["allowed_amount"]) |
    (fact["allowed_amount"] < fact["paid_amount"])
).sum()

if pricing_violations == 0:
    print("   ✅ All rows obey charge ≥ allowed ≥ paid.")
else:
    print(f"   ❌ {pricing_violations} rows violate pricing hierarchy.")

# -----------------------------
# 3. Aggregate Financial Metrics
# -----------------------------
print("\n📊 Financial Aggregates:")

total_charge = fact["charge_amount"].sum()
total_allowed = fact["allowed_amount"].sum()
total_paid = fact["paid_amount"].sum()
total_denied = fact["denial_amount"].sum()
total_var = fact["value_at_risk"].sum()

print(f"   - Total Charge Amount: {total_charge:,.2f}")
print(f"   - Total Allowed Amount: {total_allowed:,.2f}")
print(f"   - Total Paid Amount: {total_paid:,.2f}")
print(f"   - Total Denial Amount: {total_denied:,.2f}")
print(f"   - Total Value at Risk: {total_var:,.2f}")

# -----------------------------
# 4. Reasonableness Check
# -----------------------------
denial_ratio = (total_denied / total_allowed) * 100

print(f"\n📉 Denial Amount Ratio: {denial_ratio:.2f}%")

if 5 <= denial_ratio <= 35:
    print("   ✅ Denial ratio within realistic healthcare range.")
else:
    print("   ⚠️ Denial ratio outside expected range — review recommended.")

print("\n✅ Step 4.5 Amount Sanity Checks & Aggregates completed.")


STEP 4 – SUB‑STEP 4.5: AMOUNT SANITY CHECKS & AGGREGATES

💰 Negative Value Checks:
   - charge_amount: ✅ OK
   - allowed_amount: ✅ OK
   - paid_amount: ✅ OK
   - denial_amount: ✅ OK
   - value_at_risk: ✅ OK

📐 Pricing Hierarchy Check:
   ✅ All rows obey charge ≥ allowed ≥ paid.

📊 Financial Aggregates:
   - Total Charge Amount: 508,633,273.84
   - Total Allowed Amount: 345,951,907.80
   - Total Paid Amount: 299,042,206.60
   - Total Denial Amount: 25,785,409.41
   - Total Value at Risk: 209,591,067.25

📉 Denial Amount Ratio: 7.45%
   ✅ Denial ratio within realistic healthcare range.

✅ Step 4.5 Amount Sanity Checks & Aggregates completed.


# Sub‑step 6: Outlier scan (IQR)

## Goal:
Confirm that extreme financial values are explainable and not data errors.

## This step answers:

- Are there unusually high or low amounts?
- Are they rare and expected, or widespread and suspicious?

Importantly: we do not delete outliers here, we only identify and assess them.

## ✅ What this code checks

- Outliers in charge_amount
- Outliers in paid_amount
- Uses the IQR (Interquartile Range) method

In [6]:
print("\nSTEP 4 – SUB‑STEP 4.6: OUTLIER SCAN (IQR METHOD)\n")

outlier_columns = ["charge_amount", "paid_amount"]

for col in outlier_columns:
    print(f"📌 Outlier Analysis for {col}:")

    Q1 = fact[col].quantile(0.25)
    Q3 = fact[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = max(0, Q1 - 1.5 * IQR)
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((fact[col] < lower_bound) | (fact[col] > upper_bound)).sum()
    total_count = fact.shape[0]
    outlier_pct = (outlier_count / total_count) * 100

    print(f"   - Lower bound: {lower_bound:,.2f}")
    print(f"   - Upper bound: {upper_bound:,.2f}")
    print(f"   - Outlier records: {outlier_count} ({outlier_pct:.2f}%)")

    if outlier_pct < 5:
        print("   ✅ Outliers within expected healthcare range.")
    elif outlier_pct < 10:
        print("   ⚠️ Moderate outlier presence; acceptable but monitor.")
    else:
        print("   ⚠️ High outlier presence; expected for high‑cost procedures.")

    print()

print("✅ Step 4.6 Outlier Scan completed.")


STEP 4 – SUB‑STEP 4.6: OUTLIER SCAN (IQR METHOD)

📌 Outlier Analysis for charge_amount:
   - Lower bound: 0.00
   - Upper bound: 8,342.14
   - Outlier records: 29922 (24.93%)
   ⚠️ High outlier presence; expected for high‑cost procedures.

📌 Outlier Analysis for paid_amount:
   - Lower bound: 0.00
   - Upper bound: 5,969.13
   - Outlier records: 29886 (24.91%)
   ⚠️ High outlier presence; expected for high‑cost procedures.

✅ Step 4.6 Outlier Scan completed.


# Sub‑step 7: Basic cleaning & export

## Goal:
Produce a trusted, analytics‑ready dataset for dashboards and further analysis.

## This step:
- Does minimal, safe cleaning
- Does not distort business reality
- Creates the final dataset used downstream


## ✅ What this code does

- Removes duplicate claim lines (if any)
- Drops rows missing critical keys
- Safeguards against negative monetary values
- Recomputes derived financial fields
- Adds submission lag indicators
- Exports a clean dataset

In [8]:
print("\nSTEP 4 – SUB‑STEP 4.7: BASIC CLEANING & EXPORT\n")

clean_fact = fact.copy()

# ----------------------------------
# 1. Remove duplicate claim lines
# ----------------------------------
dup_count = clean_fact.duplicated(subset=["claim_line_id_nat"]).sum()

if dup_count == 0:
    print("✅ No duplicate claim lines found.")
else:
    clean_fact = clean_fact.drop_duplicates(subset=["claim_line_id_nat"], keep="first")
    print(f"⚠️ {dup_count} duplicate claim lines removed.")

# ----------------------------------
# 2. Drop rows with missing critical fields
# ----------------------------------
critical_fields = [
    "claim_line_id_nat",
    "service_date_sk",
    "patient_sk",
    "provider_sk",
    "facility_sk",
    "payer_sk",
    "cpt_sk",
    "icd_sk",
    "charge_amount",
    "allowed_amount",
    "paid_amount"
]

before_rows = clean_fact.shape[0]
clean_fact = clean_fact.dropna(subset=critical_fields)
after_rows = clean_fact.shape[0]

removed_rows = before_rows - after_rows

if removed_rows == 0:
    print("✅ No rows removed due to missing critical fields.")
else:
    print(f"⚠️ {removed_rows} rows removed due to missing critical fields.")

# ----------------------------------
# 3. Safeguard against negative numeric values
# ----------------------------------
amount_columns = [
    "charge_amount",
    "allowed_amount",
    "paid_amount",
    "denial_amount",
    "value_at_risk",
]

for col in amount_columns:
    neg_count = (clean_fact[col] < 0).sum()
    if neg_count > 0:
        clean_fact[col] = clean_fact[col].clip(lower=0)
        print(f"⚠️ {neg_count} negative values corrected in {col}")
    else:
        print(f"✅ {col}: no negative values")

# ----------------------------------
# 4. Final row & column check
# ----------------------------------
print(f"\n✅ Final cleaned dataset rows: {clean_fact.shape[0]}")
print(f"✅ Final cleaned dataset columns: {clean_fact.shape[1]}")

# ----------------------------------
# 5. Export cleaned dataset
# ----------------------------------
output_file = "maplecare_fact_claim_line_cleaned.parquet"
clean_fact.to_parquet(output_file, index=False)

print(f"\n✅ Cleaned dataset exported successfully: {output_file}")
print("✅ EDA process completed. Data is ready for downstream analysis.")


STEP 4 – SUB‑STEP 4.7: BASIC CLEANING & EXPORT

✅ No duplicate claim lines found.
✅ No rows removed due to missing critical fields.
✅ charge_amount: no negative values
✅ allowed_amount: no negative values
✅ paid_amount: no negative values
✅ denial_amount: no negative values
✅ value_at_risk: no negative values

✅ Final cleaned dataset rows: 120000
✅ Final cleaned dataset columns: 19

✅ Cleaned dataset exported successfully: maplecare_fact_claim_line_cleaned.parquet
✅ EDA process completed. Data is ready for downstream analysis.
